# Test toàn bộ pipeline Agentic RAG (end-to-end)

Notebook này tách hệ thống thành **6 trạm dừng** để quan sát rõ từng giai đoạn biến đổi của một câu hỏi:

```
(1) Câu hỏi  →  (2) Embedding (vector 768d)  →  (3) Qdrant similarity + BM25 + RRF
             →  (4) Agent Analyzer (phân loại + chuẩn hoá + mở rộng truy vấn)
             →  (5) LangGraph Agent (routing → legal_rag / chit_chat / web / out_of_scope)
             →  (6) Câu trả lời + trích dẫn
```

**Tiền điều kiện**
- Qdrant đang chạy ở `localhost:6333` (collection `Traffic_Law_Hybrid` đã được index).
- File `.env` ở repo root chứa `API_KEY` (Gemini) và (tuỳ chọn) `TAVILY_API_KEY` cho nhánh web fallback.
- Venv `~/venv/LLM_Agentic` có sẵn các phụ thuộc trong `requirements.txt`.


## 0. Cấu hình hiển thị + nạp `.env`

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.jp-OutputArea-child { max-height: unset !important; } .jp-OutputArea-output { max-height: unset !important; }</style>"))

import os, sys
from pathlib import Path

# tests/ → traffic_rag/  (để import được `source.*`)
BASE_DIR = Path().resolve().parent if Path().resolve().name in ("notebooks", "tests") else Path().resolve()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
if str(BASE_DIR / "source") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "source"))

# Nạp .env từ repo root hoặc thư mục cha
for parent in [Path().resolve(), *Path().resolve().parents]:
    cand = parent / ".env"
    if cand.exists():
        for line in cand.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip().strip('"').strip("'")
        print(f"[OK] Nạp env từ {cand}")
        break

# Map API_KEY → GOOGLE_API_KEY / GEMINI_API_KEY (tuỳ convention dự án)
shared = os.environ.get("API_KEY")
if shared:
    os.environ.setdefault("GOOGLE_API_KEY", shared)
    os.environ.setdefault("GEMINI_API_KEY", shared)

print("GOOGLE_API_KEY:", "OK" if os.environ.get("GOOGLE_API_KEY") else "MISSING")
print("TAVILY_API_KEY:", "OK" if os.environ.get("TAVILY_API_KEY") else "MISSING (nhánh web fallback sẽ tắt)")

## 1. Câu hỏi đầu vào

Bạn có thể đổi `QUESTION` ở đây — toàn bộ notebook sẽ chạy theo câu hỏi này.

In [ ]:
QUESTION = "Vượt đèn đỏ khi điều khiển ô tô bị phạt bao nhiêu tiền và trừ bao nhiêu điểm GPLX?"
print("=" * 80)
print("Câu hỏi gốc:")
print(" ", QUESTION)
print("=" * 80)

## 2. Bước 1 — Câu hỏi → Vector Embedding

Dự án dùng `intfloat/multilingual-e5-base` (768 chiều). Với họ E5, **truy vấn phải có prefix `"query: "`** trước khi encode (đó là yêu cầu của model). Cell này encode trực tiếp, in ra **shape, norm, 16 chiều đầu** để bạn nhìn thấy vector thật.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "intfloat/multilingual-e5-base"
print(f"Đang nạp model embedding: {EMBED_MODEL} ...")
embedder = SentenceTransformer(EMBED_MODEL)

query_text = f"query: {QUESTION}"   # prefix bắt buộc cho e5
vec = embedder.encode(query_text, normalize_embeddings=False)
vec = np.asarray(vec, dtype=np.float32)

print("\nshape :", vec.shape)
print("dtype :", vec.dtype)
print("L2 norm:", float(np.linalg.norm(vec)))
print("\n16 chiều đầu của vector:")
print(np.round(vec[:16], 4).tolist())

## 3. Bước 2 — So sánh vector với Qdrant (similarity thuần)

Truy vấn **dense** trực tiếp lên collection `Traffic_Law_Hybrid` để thấy **cosine similarity** (Qdrant trả ra điểm cosine vì collection được build với `Distance.COSINE`). Đây là tầng `dense` trước khi RRF với BM25.

In [ ]:
from qdrant_client import QdrantClient

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333
COLLECTION  = "Traffic_Law_Hybrid"

client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
info = client.get_collection(COLLECTION)
print(f"Collection: {COLLECTION}")
print(f"  Vectors: {info.points_count} điểm")
print(f"  Config : {info.config.params.vectors}")

TOP_K = 5
hits = client.search(
    collection_name=COLLECTION,
    query_vector=vec.tolist(),
    limit=TOP_K,
    with_payload=True,
)

print(f"\nTop-{TOP_K} chunks gần nhất theo cosine (chỉ dense, chưa RRF):\n")
for i, h in enumerate(hits, 1):
    md = h.payload or {}
    print(f"[{i}] cosine={h.score:.4f}  id={h.id}  doc={md.get('doc_id','?')}  Điều {md.get('dieu','?')}")
    text = (md.get("content") or md.get("text") or "")[:160].replace("\n", " ")
    print(f"    {text}...\n")

## 4. Bước 3 — Hybrid Retriever (Dense + BM25 + RRF)

`TrafficHybridRetriever` của dự án kết hợp:
- **Dense**: kết quả Qdrant ở bước 3.
- **Sparse**: BM25 in-memory trên cùng tập tài liệu.
- **Fusion**: Reciprocal Rank Fusion (`rrf_k=60`).

So sánh kết quả với cell trước để thấy BM25 "kéo" được những điều luật mà dense bỏ sót khi câu hỏi dùng khẩu ngữ (vd. "vượt đèn đỏ" vs cụm pháp lý "không chấp hành hiệu lệnh đèn tín hiệu").

In [ ]:
from rag_core import TrafficHybridRetriever

retriever = TrafficHybridRetriever()
chunks = retriever.get_relevant_chunks(QUESTION, top_k=5)

print(f"Hybrid retriever trả {len(chunks)} chunks (đã dedup):\n")
for i, c in enumerate(chunks[:8], 1):
    md = c.metadata
    print(f"[{i}] RRF={c.score:.4f}  doc={md.get('doc_id','?')}  Điều {md.get('dieu','?')}")
    print(f"    {c.content[:160].replace(chr(10),' ')}...\n")

## 5. Bước 4 — Agent Analyzer (phân loại + chuẩn hoá + mở rộng query)

Trước khi đi vào graph, `analyzer_node` gọi LLM 1 lần duy nhất để:
1. **Phân loại** vào `legal_rag` / `chit_chat` / `web_legal_search` / `out_of_scope`.
2. **Chuẩn hoá** (`standalone_query`): viết lại câu độc lập, giải tham chiếu ("xe đó" → "xe ô tô con"...).
3. **Mở rộng** (`expanded_query`): thêm thuật ngữ pháp lý chính thức để retriever tìm trúng — đây mới là chuỗi đem đi search ở bước sau.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from agent.nodes import make_analyzer_node

GEN_MODEL = "gemini-3.1-flash-lite-preview"

llm = ChatGoogleGenerativeAI(
    model=GEN_MODEL,
    temperature=0.1,
    google_api_key=os.environ["GOOGLE_API_KEY"],
    request_timeout=30,
)

analyzer = make_analyzer_node(llm)
analyzer_state = {"query": QUESTION, "chat_history": []}
out = analyzer(analyzer_state)

print("--- KẾT QUẢ ANALYZER ---")
print("category         :", out.get("category"))
print("raw_query        :", out.get("raw_query"))
print("standalone_query :", out.get("query"))
print("expanded_query   :", out.get("expanded_query"))

### 5.1 — So sánh retrieval với `expanded_query`

Đem `expanded_query` từ bước 5 đi search lại để xem mức độ "trúng đích" cải thiện ra sao so với câu hỏi thô ở bước 4.

In [ ]:
expanded = out.get("expanded_query") or QUESTION
chunks_exp = retriever.get_relevant_chunks(expanded, top_k=5)

print(f"Top-5 với expanded_query (RRF):\n")
for i, c in enumerate(chunks_exp, 1):
    md = c.metadata
    print(f"[{i}] RRF={c.score:.4f}  doc={md.get('doc_id','?')}  Điều {md.get('dieu','?')}")
    print(f"    {c.content[:160].replace(chr(10),' ')}...\n")

## 6. Bước 5 — Build Generator + LangGraph Agent

Lắp đầy đủ graph như API production (`api/main.py`), nhưng **tắt HITL interrupt** (`enable_hitl_interrupt=False`) để chạy end-to-end trong notebook không cần phê duyệt thủ công.

Nếu không có `TAVILY_API_KEY` ta truyền `tavily_tool=None` — graph vẫn chạy được nhánh `legal_rag` / `chit_chat` / `out_of_scope`, chỉ là khi router đưa vào `web_search` thì sẽ raise.

In [ ]:
from rag_core import LegalAnswerGenerator
from agent import build_graph, TavilySearchTool

generator = LegalAnswerGenerator(provider="google", model=GEN_MODEL)

tavily = None
if os.environ.get("TAVILY_API_KEY"):
    try:
        tavily = TavilySearchTool()
        print("[OK] Tavily web-search tool đã sẵn sàng.")
    except Exception as e:
        print(f"[WARN] Không khởi tạo được Tavily: {e}")
else:
    print("[INFO] TAVILY_API_KEY không có — nhánh web_search sẽ không khả dụng.")

graph = build_graph(
    retriever=retriever,
    generator=generator,
    llm=llm,
    tavily_tool=tavily,
    checkpoint_db=str(BASE_DIR / "checkpoints" / "notebook_test.db"),
    enable_hitl_interrupt=False,   # chạy thẳng, không pause để approve
)
print("[OK] Graph compiled.")

## 7. Bước 6 — Chạy graph end-to-end

`graph.invoke(...)` sẽ chạy chuỗi node: `analyzer → (route) → legal_rag → END` và trả về toàn bộ `AgentState`. Mỗi `thread_id` ứng với một phiên hội thoại được lưu trong SQLite checkpointer (giống production).

In [ ]:
import uuid

config = {"configurable": {"thread_id": f"nb-{uuid.uuid4().hex[:8]}"}}

final_state = graph.invoke(
    {"query": QUESTION, "chat_history": []},
    config=config,
)

print("=" * 80)
print("thread_id   :", config["configurable"]["thread_id"])
print("category    :", final_state.get("category"))
print("refused     :", final_state.get("refused"))
print("#chunks     :", len(final_state.get("chunks") or []))
print("model       :", final_state.get("model_info"))
print("=" * 80)
print("\n========= CÂU TRẢ LỜI =========\n")
print(final_state.get("answer") or "(rỗng)")
print("\n========= TRÍCH DẪN =========\n")
for s in (final_state.get("sources") or []):
    parts = [s.get("doc_id", "?"), f"Điều {s.get('dieu','?')}"]
    if s.get("khoan") is not None: parts.append(f"Khoản {s['khoan']}")
    if s.get("diem")  is not None: parts.append(f"Điểm {s['diem']}")
    print("  -", " | ".join(parts), "—", s.get("ten_van_ban", ""))

## 8. Trace từng node (debug)

Dùng `graph.stream(...)` thay cho `invoke` để **xem state tăng dần qua từng node**. Hữu ích khi muốn biết node nào đã ghi field nào, hoặc query đi qua đường nào trong 4 nhánh.

In [ ]:
config2 = {"configurable": {"thread_id": f"nb-trace-{uuid.uuid4().hex[:8]}"}}

print(f"[TRACE] thread_id={config2['configurable']['thread_id']}\n")
for step in graph.stream(
    {"query": QUESTION, "chat_history": []},
    config=config2,
    stream_mode="updates",
):
    for node_name, delta in step.items():
        keys = list(delta.keys()) if isinstance(delta, dict) else type(delta).__name__
        print(f"→ NODE: {node_name}")
        print(f"   updates keys: {keys}")
        if isinstance(delta, dict):
            for k in ("category", "expanded_query", "refused", "answer"):
                if k in delta:
                    val = delta[k]
                    if isinstance(val, str) and len(val) > 200:
                        val = val[:200] + "…"
                    print(f"   {k}: {val}")
        print()

## 9. Test các nhánh routing khác

Cùng một graph, khác câu hỏi → router đưa vào các nhánh khác nhau:
- `chit_chat` (chào hỏi)
- `out_of_scope` (không liên quan giao thông)
- `legal_rag` (mặc định cho luật giao thông VN)

*Ghi chú:* nhánh `web_legal_search` cần `TAVILY_API_KEY` thật mới chạy được; nếu không có, để `enable_hitl_interrupt=False` thì cũng không pause, nhưng tool gọi sẽ thất bại — bỏ qua case đó nếu chưa cấu hình.

In [ ]:
import time

ROUTING_CASES = [
    ("chit_chat (kỳ vọng)",   "Xin chào, bạn là ai?"),
    ("out_of_scope (kỳ vọng)", "Cho tôi công thức nấu phở bò"),
    ("legal_rag (kỳ vọng)",   "Nồng độ cồn vượt 0,4 mg/lít khí thở khi đi xe máy phạt bao nhiêu?"),
]

for label, q in ROUTING_CASES:
    print("=" * 80)
    print(f"[CASE] {label}")
    print(f"Q: {q}")
    cfg = {"configurable": {"thread_id": f"nb-route-{uuid.uuid4().hex[:8]}"}}
    try:
        st = graph.invoke({"query": q, "chat_history": []}, config=cfg)
        print(f"category : {st.get('category')}")
        ans = st.get("answer") or ""
        print(f"answer   : {ans[:300]}{'…' if len(ans) > 300 else ''}")
    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")
    print()
    time.sleep(4)  # tránh đụng rate limit free tier 15 RPM

## 10. Tóm tắt — bạn vừa quan sát điều gì?

| Bước | Đầu vào | Đầu ra | File nguồn |
|---|---|---|---|
| 1 | Câu hỏi text | Vector 768d (E5) | `source/rag_core/retriever.py` |
| 2 | Vector | Top-k cosine từ Qdrant | (Qdrant client) |
| 3 | Câu hỏi | Top-k chunks RRF (Dense+BM25) | `source/rag_core/retriever.py` |
| 4 | Câu hỏi + history | category, standalone, expanded | `source/agent/nodes.py:make_analyzer_node` |
| 5 | AgentState | Câu trả lời + sources | `source/agent/graph.py:build_graph` |
| 6 | Stream updates | Trace từng node | `graph.stream(...)` |

**Khi nào dùng notebook này lại:**
- Sau khi đổi model embedding hoặc cấu hình RRF: đổi `EMBED_MODEL`/`rrf_k` ở đầu, chạy lại bước 2–4 và so sánh top-k.
- Sau khi sửa prompt analyzer: chạy lại bước 5 với câu hỏi khó (đa ý, follow-up có "xe đó"...) để check expanded_query.
- Khi nghi nhánh routing sai: bước 9 là test suite tối thiểu cho 3/4 nhánh.
